# Tutorial 6 — Advanced Features

Tutorials 1–5 cover everything needed to write a knowledge base, with or without structural dimensions, and to train it. This tutorial presents features that become relevant once that workflow is in place. The four parts are independent: each begins with its own setup cell, so they can be read in any order.

- **Part A — Language and evaluation:** infix symbols and the evaluation cache.
- **Part B — Structure:** arguments without a consumed axis, arguments with different dimensions, soft and unary relations, relations as atoms, soft guards as attention, structural variables as terms, several dimensions, and definitions with structural variables. It assumes Tutorial 5.
- **Part C — Training:** multi-objective optimisation and PyTorch Lightning. It reuses the disc example of Tutorial 4.
- **Part D — Evaluation and debugging:** crisp Boolean evaluation with `Logic.classical()` and type-constraint checks with `validate_type_constraint`. It reuses the animals of Tutorials 1–2.

Two sections depend on optional packages: `pip install "sltn[multiobjective]"` (TorchJD) and `pip install "sltn[lightning]"`. If a package is missing, the corresponding cell prints a message and is skipped.

In [ ]:
import math
import os
import tempfile

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import sltn
from sltn import KB, Interpretation, Logic, Signature, kb_backward, kb_describe, kb_eval
from sltn.interpretation.operators import AggregMax, AggregMin, AggregPMean, AggregPMeanError

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

# Part A — Language and evaluation

## A.1 — Infix symbols

A binary function or predicate declared with `infix=True` is written between its arguments. Its name may then be a sequence of symbol characters such as `+` or `<`. It is grounded like any other symbol, e.g. `interp["+"] = …`. The parser applies the following rules.

- Infix **functions** bind more tightly than infix **predicates**, which bind more tightly than the connectives. `a + b < c & φ` is therefore parsed as `((a + b) < c) & φ`.
- All infix functions share a single left-associative precedence level, so `a + b + c` is `(a + b) + c`.
- Infix predicates are not chainable. `a < b < c` is an error, and the intended formula is written `a < b & b < c`.
- A name may not contain whitespace or any of `()[]{},:|`. The lexer uses longest match, so `<` does not conflict with `<->`.

In [ ]:
sig_inf = Signature("numbers")
sig_inf.sort("Num")
sig_inf.variable("a", "Num")
sig_inf.variable("b", "Num")
sig_inf.function("+", ["Num", "Num"], "Num", infix=True)
sig_inf.predicate("<", ["Num", "Num"], infix=True)

interp_inf = Interpretation(sig_inf)
interp_inf["Num"] = sltn.Type("number", shape=(1,))
interp_inf["+"] = lambda u, v: u + v
interp_inf["<"] = lambda u, v: torch.sigmoid(5 * (v - u))      # soft "less than"
interp_inf["a"] = torch.tensor([[0.0], [1.0], [2.0]])
interp_inf["b"] = torch.tensor([[1.0], [3.0]])

phi = sig_inf.parse("a < a + b -> not (a + b < a)")
print(phi)
print(interp_inf(phi).value.squeeze(-1))

try:
    sig_inf.parse("a < b < a")
except ValueError as e:
    print("\na < b < a\n  ->", e)

## A.2 — The evaluation cache

Within a single call `interp(phi)`, each repeated sub-formula or sub-term is evaluated **once** and its result reused. The cache is discarded at the end of the call, so new data is always taken into account. It is enabled by default (`interp.use_cache = True`) and changes only the number of grounding calls, not values or gradients. The one exception is dropout in training mode: a repeated atom shares a single dropout mask, which keeps the formula consistent (`InDisc(x) <-> InDisc(x)` evaluates to exactly 1).

The following cell counts the calls of a predicate in a formula that contains it three times.

In [ ]:
sig2 = Signature("quadrants")
sig2.sort("Point")
sig2.variable("p", "Point")
sig2.predicate("IsRight", ["Point"])
interp2 = Interpretation(sig2)
interp2["Point"] = sltn.Type("R2", shape=(2,))
interp2["p"] = torch.rand(6, 2)

calls = {"IsRight": 0}

class CountedIsRight(nn.Module):
    def forward(self, a):
        calls["IsRight"] += 1
        return (a[:, [0]] > 0.5).float()

interp2["IsRight"] = CountedIsRight()
formula = "IsRight(p) & IsRight(p) & IsRight(p)"

for use_cache in [True, False]:
    interp2.use_cache = use_cache
    calls["IsRight"] = 0
    with torch.no_grad():
        interp2(formula)
    print(f"use_cache={use_cache}:  IsRight called {calls['IsRight']} time(s) for {formula}")

interp2.use_cache = True

# Part B — Structure

This part continues the hospital ward example of Tutorial 5. The setup cell rebuilds its signature and groundings: four patients, five days, the `Fever` predicate and the `next` / `before` relations.

In [ ]:
temps = torch.tensor([
    [39.0, 38.6, 38.0, 37.2, 36.8],   # 0: recovers
    [37.0, 37.4, 38.2, 38.9, 39.3],   # 1: gets worse
    [36.8, 36.9, 36.7, 36.9, 36.8],   # 2: never has fever
    [38.6, 37.1, 36.9, 38.4, 38.8],   # 3: recovers, then relapses
])

sig = Signature("ward")
sig.sort("Reading")
sig.dimension("Day")
sig.structural_variable("d", "Day")
sig.variable("x", "Reading", dims=["Day"])
sig.predicate("Fever", ["Reading"])
sig.structural_relation("next", ["Day", "Day"])
sig.structural_relation("before", ["Day", "Day"])

interp = Interpretation(sig)
interp["Reading"] = sltn.Type("temperature", shape=(1,))
interp["x"] = temps.unsqueeze(-1)            # [4 patients, 5 days, 1]
interp["Fever"] = lambda v: torch.sigmoid(3 * (v - 37.8))
days = torch.arange(5)
interp["next"] = (days.unsqueeze(1) + 1 == days.unsqueeze(0)).float()
interp["before"] = (days.unsqueeze(1) < days.unsqueeze(0)).float()

## B.1 — An argument without the consumed axis

A predicate that consumes a dimension can take further arguments alongside the sequence, for instance a target temperature to compare a patient's stay with. The target has no day axis, so it is declared as a plain constant.

**Every argument receives all the consumed axes of the symbol.** `Day` is consumed, so `target` arrives as `[B, Day, 1]`, with its value repeated on every day, exactly like `x[d]`. The two arguments therefore have the same shape and can be combined with ordinary element-wise operations:

In [ ]:
sig.constant("target", "Reading")
sig.predicate("BelowOnAverage", ["Reading", "Reading"], input_dims=["Day"])

def below_on_average(v, c):                      # v, c: [B, Day, 1]
    print("  BelowOnAverage receives", tuple(v.shape), tuple(c.shape))
    return torch.sigmoid(3 * (c - v).mean(dim=1))   # [B, 1]

interp["target"] = torch.tensor([37.5])
interp["BelowOnAverage"] = below_on_average
print(interp("BelowOnAverage(x[d], target)").value.squeeze(-1))

Consumed axes are aligned by name, like free axes: arguments that share an axis name are matched position by position. Each dimension in `input_dims` must occur on at least one argument, and on no more distinct axes than it is listed: with `input_dims=["Day"]`, `P(x[d], y[d1])` is rejected, since it has two `Day` axes.

When the grounding reduces the consumed axis, it must reduce it in every argument: `c - v.mean(dim=1)` mixes a `[B, Day, 1]` with a `[B, 1]` tensor and fails. Test groundings with batch and dimension sizes that differ, so that a shape mismatch raises an error instead of broadcasting silently.

## B.2 — Arguments with different dimensions

The arguments of a symbol can have different structural dimensions, e.g. `x[d]` over `Day` and `y[s]` over `Site`.

- **Pointwise** (no `input_dims`): both axes stay free and the grounding is applied to every day–site combination, as distinct variables are crossed.
- **Consuming** (`input_dims=["Day", "Site"]`): `input_dims` applies to the symbol as a whole, so **every** argument receives both axes, in that order. `Site` is copied into `x` and `Day` into `y`; as in B.1, the grounding combines the arguments element-wise or reduces each of them.

Batch sizes (2 and 3) and dimension sizes (4 and 5) differ below, so each shape is unambiguous.

> **Note.** `input_dims` currently applies to the symbol as a whole. Future versions will likely allow declaring the consumed dimensions per argument, so that each argument receives only its own axes.

In [ ]:
mixed_sig = Signature("mixed axes")
mixed_sig.sort("Reading")
mixed_sig.dimension("Day")
mixed_sig.dimension("Site")
mixed_sig.structural_variable("d", "Day")
mixed_sig.structural_variable("s", "Site")
mixed_sig.variable("x", "Reading", dims=["Day"])
mixed_sig.variable("y", "Reading", dims=["Site"])

mixed_sig.predicate("Compare", ["Reading", "Reading"])
mixed_sig.function("Add", ["Reading", "Reading"], "Reading")
mixed_sig.predicate("CompareMeans", ["Reading", "Reading"], input_dims=["Day", "Site"])
mixed_sig.function("AddMeans", ["Reading", "Reading"], "Reading", input_dims=["Day", "Site"])

mixed_interp = Interpretation(mixed_sig)
mixed_interp["Reading"] = sltn.Type("MixedReading", shape=(1,))
x_data = torch.arange(8, dtype=torch.float32).reshape(2, 4, 1) / 10
y_data = torch.arange(15, dtype=torch.float32).reshape(3, 5, 1) / 10
mixed_interp["x"] = x_data
mixed_interp["y"] = y_data

def compare(a, b):
    print("  Compare receives", tuple(a.shape), tuple(b.shape))
    return torch.sigmoid(a - b)

def compare_means(a, b):
    print("  CompareMeans receives", tuple(a.shape), tuple(b.shape))
    return torch.sigmoid((a - b).mean(dim=(1, 2)))

def add_means(a, b):
    print("  AddMeans receives", tuple(a.shape), tuple(b.shape))
    return a.mean(dim=(1, 2)) + b.mean(dim=(1, 2))   # the copies do not change the mean

mixed_interp["Compare"] = compare
mixed_interp["Add"] = lambda a, b: a + b
mixed_interp["CompareMeans"] = compare_means
mixed_interp["AddMeans"] = add_means

point_pred = mixed_interp("Compare(x[d], y[s])")
point_fn = mixed_interp("Add(x[d], y[s])")
whole_pred = mixed_interp("CompareMeans(x[d], y[s])")
whole_fn = mixed_interp("AddMeans(x[d], y[s])")
for name, result in [("Compare", point_pred), ("Add", point_fn),
                     ("CompareMeans", whole_pred), ("AddMeans", whole_fn)]:
    print(name, "axes", result.axis_names, "shape", tuple(result.value.shape))

## B.3 — Soft weights and unary relations

A relation with values strictly between 0 and 1 weights positions instead of including or excluding them, so the aggregation becomes a weighted mean. A relation may also have a single argument. `Recent(d)` assigns more weight to later days, so "no fever lately" depends mainly on the end of the stay:

In [ ]:
sig.structural_relation("Recent", ["Day"])
interp["Recent"] = torch.tensor([0.1, 0.2, 0.4, 0.7, 1.0])

lately = interp("∀ d | Recent(d): ¬Fever(x[d])").value.squeeze(-1)
uniform = interp("∀ d: ¬Fever(x[d])").value.squeeze(-1)
for k in range(4):
    print(f"patient {k}: no fever lately {lately[k]:.2f}   no fever ever {uniform[k]:.2f}")

With a guard, the quantifier computes a **weighted** pMeanError, $1 - \sqrt{\sum_d w_d (1 - \varphi_d)^2 / \sum_d w_d}$, with $w_d$ = `Recent(d)`. The weights change how much each day counts, not its truth value: patient 0, whose fever is at the start of the stay, scores higher on "no fever lately" than on "no fever ever", while patients 1 and 3, with fever at the end, score lower.

## B.4 — Relations as atoms

A relation is itself a formula with its own axes, so it can also appear in the body of a quantified formula. The result differs from the guarded version:

In [ ]:
guard = interp("∀ d, d1 | next(d, d1): (Fever(x[d]) -> Fever(x[d1]))").value.squeeze(-1)
atom = interp("∀ d, d1: (next(d, d1) -> (Fever(x[d]) -> Fever(x[d1])))").value.squeeze(-1)
print("as a guard:", guard)
print("as an atom:", atom)

With the atom, the quantifier aggregates all 25 pairs of days, and the 21 pairs that are not consecutive are true, since `0 -> …` is 1. They raise every value towards 1, e.g. from 0.67 to 0.87 for patient 0, whose fever stops.

A guard therefore determines which positions the quantifier ranges over, whereas an atom is part of what the formula asserts, and its truth degree is combined with the others by the connectives. An atom is needed when the relation is part of the claim, for instance as a conclusion, which a guard cannot express. The following cell evaluates "every day with fever comes before every day without fever".

In [ ]:
before_fever = interp("∀ d, d1: ((Fever(x[d]) & ¬Fever(x[d1])) -> before(d, d1))").value.squeeze(-1)
for k in range(4):
    print(f"patient {k}: {before_fever[k]:.2f}")

`before` is here the conclusion of the implication. Patient 0, whose days with fever precede the days without, satisfies the formula. Patients 1 and 3, who have days without fever before days with fever, violate it.

**Current limitation.** A relation cannot take the same structural variable twice. `next(d, d)` raises `Axis names must be unique`, whereas `Fever(x[d]) & Fever(x[d])` is valid. For `next` this is irrelevant, since no day follows itself, but it matters for relations whose diagonal is meaningful. On a graph of people, "nobody is their own friend" concerns `friends(p, p)`. It is written with an additional relation `same(p, p1)`, grounded with the identity matrix, as `∀ p, p1 | same(p, p1): ¬friends(p, p1)`.

## B.5 — Soft guards as attention

A binary soft guard behaves like attention pooling. With `AggregPMean(p=1)`, `∃ d1 | w(d, d1): φ(d1)` computes

$$\sum_{d_1} \frac{w(d, d_1)}{\sum_{d_1'} w(d, d_1')}\, \varphi(d_1),$$

a weighted mean of $\varphi$ over the days $d_1$, separately for each day $d$: $d$ plays the query, $d_1$ the keys and $\varphi$ the values. Since the quantifier normalises the weights, any non-negative guard works, without an explicit softmax. With $p > 1$ the result is a weighted power mean instead.

`Near(d, d1)` weights each day by its distance to the others, so each day attends mostly to itself and its neighbours. The result keeps the free axis `d`: a fever level per day, smoothed over nearby days.

In [ ]:
sig.structural_relation("Near", ["Day", "Day"])
days = torch.arange(5).float()
interp["Near"] = torch.softmax(-(days.unsqueeze(1) - days.unsqueeze(0)).abs(), dim=1)   # each row sums to 1
print(interp["Near"].round(decimals=2))

interp.logic["∃,Day"] = AggregPMean(p=1)                 # plain weighted mean
attended = interp("∃ d1 | Near(d, d1): Fever(x[d1])")
print("axes:", attended.axis_names)

print(attended.value.squeeze(-1).round(decimals=2))

The quantifier normalises the weights, so any values in [0, 1] can be used, and they may differ between patients. A guard can be any formula, not only a relation. In the following cell each day attends to the days **of the same patient** with a similar reading, through a predicate on two readings. The relapse of patient 3 on days 3–4 is then less mixed with the days 1–2 without fever than with `Near`.

In [ ]:
sig.predicate("Similar", ["Reading", "Reading"])
interp["Similar"] = lambda v, w: torch.exp(-(v - w) ** 2)  # could be an nn.Module

same_course = interp("∃ d1 | Similar(x[d], x[d1]): Fever(x[d1])")
print("axes:", same_course.axis_names)
print(same_course.value.squeeze(-1).round(decimals=2))
interp.logic["∃,Day"] = AggregPMean(p=2)

**Learned weights.** A guard grounded with an `nn.Module` is trained together with the rest of the interpretation. It still determines which positions the quantifier ranges over.

**Only pMean and pMeanError use the weights.** `AggregMin` and `AggregMax` treat a guard as a hard filter. A day with weight 0.1 counts as much as a day with weight 1, and the guard receives no gradient through them. With `Recent` from B.3, min therefore gives the same values with and without the guard.

In [ ]:
interp.logic["∀,Day"] = AggregMin()
print("min, Recent guard:", interp("∀ d | Recent(d): ¬Fever(x[d])").value.squeeze(-1))
print("min, no guard:    ", interp("∀ d: ¬Fever(x[d])").value.squeeze(-1))
interp.logic["∀,Day"] = AggregPMeanError(p=2)

## B.6 — Structural variables as terms

So far `d` has only named an axis: day 3 is simply the fourth position. In some applications the position itself carries information: the readings may be taken at irregular times, and a rule may depend on *when* a reading was taken. If the dimension is declared with a **sort**, its structural variable can be grounded with one value per index and used as a term of that sort, like a variable.

Here the visits of each patient are taken at irregular hours since admission. `Early(v)` is a predicate on the time of visit `v`, and "fever only in the first 48 hours" is `∀ v: Fever(x[v]) -> Early(v)`:

In [ ]:
clinic = Signature("clinic")
clinic.sort("Reading")
clinic.sort("Hours")                              # time since admission
clinic.dimension("Visit", sort="Hours")           # the index of a visit carries a time
clinic.structural_variable("v", "Visit")
clinic.variable("x", "Reading", dims=["Visit"])
clinic.predicate("Fever", ["Reading"])
clinic.predicate("Early", ["Hours"])              # a predicate on the time itself

ci = Interpretation(clinic)
ci["Reading"] = sltn.Type("temperature", shape=(1,))
ci["Hours"] = sltn.Type("hours", shape=(1,))
ci["x"] = temps.unsqueeze(-1)                     # the same 4 patients, 5 visits
ci["v"] = torch.tensor([[0.], [6.], [24.], [30.], [72.]])   # irregular visit times
ci["Fever"] = lambda r: torch.sigmoid(3 * (r - 37.8))
ci["Early"] = lambda h: torch.sigmoid((48 - h) / 6)          # within the first 48 hours
ci.logic["∀,Visit"] = AggregMin()

print(ci("Early(v)").value.squeeze(-1))
print(ci("∀ v: (Fever(x[v]) -> Early(v))").value.squeeze(-1))

Patients 1 and 3 have fever after 48 hours (the last visits), so the rule fails for them. With a plain index, "the first 48 hours" cannot be expressed, because the visits are not evenly spaced.

The same mechanism gives learned positional encodings: ground the sort with a `Type` of size `k` and the structural variable with an `nn.Parameter` of shape `[size, k]`.

Over a dimension without a sort, such as `Day`, a structural variable only names an axis: grounding `d` raises an error.

In [ ]:
try:
    interp["d"] = torch.arange(5.0).unsqueeze(-1)  # Day has no sort
except TypeError as e:
    print(e)

## B.7 — Several dimensions

A symbol can have several dimensions. Here the temperature is taken three times a day: `xh` has one reading per day and per hour, and the annotation names both axes in the order of `dims`.

In [ ]:
sig.dimension("Hour")
sig.structural_variable("h", "Hour")
sig.variable("xh", "Reading", dims=["Day", "Hour"])

interp["xh"] = temps.unsqueeze(-1).unsqueeze(-1) + torch.tensor([-0.4, 0.0, 0.4]).view(1, 1, 3, 1)  # [4, 5, 3, 1]
interp.logic["∃,Hour"] = AggregMax()

r = interp("Fever(xh[d, h])")
print("Fever(xh[d, h]) axes:", r.axis_names, tuple(r.value.shape))
print("fever at some hour, every day:", interp("∀ d: ∃ h: Fever(xh[d, h])").value.squeeze(-1))

- A quantifier binds structural variables of **one** dimension: `∀ d, h: ...` is an error; nest `∀ d: ∃ h: ...`.
- **Read results by axis name, not by position.** Before combining tensors, sLTN puts their axes in a canonical order: variable axes in alphabetical order, then structural axes in alphabetical order, then the domain axes. A result's axis order can therefore differ from the order of `dims` (with structural variables named `t` and `a`, a result comes out `[..., a, t, ...]`). `r.axis_names` gives the order.
- The annotation follows the order of `dims`: `xh[h, d]` is rejected, because the first axis of `xh` is a `Day`.

In [ ]:
try:
    sig.parse("Fever(xh[h, d])")
except ValueError as e:
    print(e)

## B.8 — Definitions with structural variables

The body of a definition (Tutorial 2, section 5) may contain structural variables, with two different effects.

- A structural variable **bound** in the body is not renamed, unlike an individual variable. It therefore binds the structural variables of the same name in the argument. `Always(phi)`, defined as `∀ d: phi`, expands `Always(Fever(x[d]))` to `∀ d: Fever(x[d])`, "fever on every day".
- A structural variable **free** in the body remains free in every expansion. `FeverNext(z)`, defined as `∀ d1 | next(d, d1): Fever(z[d1])`, "fever on the day after `d`", has the free axis `d`, and its denotation contains one truth degree per patient and day. On the last day, which has no successor, the guard is empty and `∀` gives 1.

A parameter can stand for a variable with dimensions, such as `x`, and the body names its axes, as in `Fever(z[d1])`. `Persists(z)` below is the rule of Tutorial 5, section 5, written once and applied to any sequence of readings.

In [ ]:
sig.define("Always", ["phi"], "∀ d: phi")
sig.define("FeverNext", ["z"], "∀ d1 | next(d, d1): Fever(z[d1])")
sig.define("Persists", ["z"], "∀ d, d1 | next(d, d1): (Fever(z[d]) -> Fever(z[d1]))")

for formula in ["Always(Fever(x[d]))", "FeverNext(x)", "Persists(x)"]:
    r = interp(formula)
    print(f"{formula:<20s} parsed as  {sig.parse(formula)}")
    print(f"{'':<20s} axes {r.axis_names}")
    print(r.value.squeeze(-1).round(decimals=2), "\n")

# Part C — Training

This part reuses the disc example of Tutorial 4: points in the unit square, a network `InDisc`, and the knowledge base "every positive point is in the disc, and no negative point is". The setup cell rebuilds it and defines a small `train` helper around `kb_backward`.

In [ ]:
n = 200
pts = torch.rand(n, 2)
labels = (pts - 0.5).pow(2).sum(1) < 0.09   # True = inside the disc
n_train = int(n * 0.8)
pts_tr, lbl_tr = pts[:n_train], labels[:n_train]
pts_te, lbl_te = pts[n_train:], labels[n_train:]

N_BATCHES, EPOCHS = 4, 200
pos_tr, neg_tr = pts_tr[lbl_tr], pts_tr[~lbl_tr]
loader_pos = DataLoader(TensorDataset(pos_tr), batch_size=math.ceil(len(pos_tr) / N_BATCHES), shuffle=True)
loader_neg = DataLoader(TensorDataset(neg_tr), batch_size=math.ceil(len(neg_tr) / N_BATCHES), shuffle=True)

sig = Signature("disc")
sig.sort("Point")
sig.variable("x_pos", "Point")
sig.variable("x_neg", "Point")
sig.predicate("InDisc", ["Point"])
kb = KB([sig.parse("forall x_pos: InDisc(x_pos)"), sig.parse("forall x_neg: not InDisc(x_neg)")])
kb_dict = kb_describe(kb)


class InDiscNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16), nn.ELU(),
            nn.Linear(16, 16), nn.ELU(),
            nn.Linear(16, 1), nn.Sigmoid(),
        )

    def forward(self, a):                 # a: [B, 2]
        return self.net(a)                # [B, 1]


def new_interp():
    i = Interpretation(sig).to(device)
    i["Point"] = sltn.Type("R2", shape=(2,))
    i["InDisc"] = InDiscNet()
    return i


def train(i, aggregator=None):
    opt = torch.optim.Adam(i.parameters(), lr=1e-2)
    for _ in range(EPOCHS):
        for (x_pos_batch,), (x_neg_batch,) in zip(loader_pos, loader_neg):
            opt.zero_grad()
            i["x_pos"], i["x_neg"] = x_pos_batch.to(device), x_neg_batch.to(device)
            kb_backward(opt, i, kb_dict, aggregator=aggregator)
            opt.step()
    return i


def accuracy(model_interp):
    with torch.no_grad():
        pred = model_interp["InDisc"](pts_te.to(device))[:, 0].cpu() > 0.5
    return (pred == lbl_te).float().mean().item()

## C.1 — Multi-objective optimisation

By default, `kb_backward` aggregates the clauses with `SatAgg` and backpropagates a single loss, `1 − SatAgg(...)`. A weak clause can then be compensated by the others. With `aggregator="upgrad"` (or `"mgda"`, `"pcgrad"`, …), each clause is treated as a separate objective: [TorchJD](https://torchjd.org/stable/) computes one gradient per clause and combines them into an update direction that does not conflict with any of them ([Jacobian descent](https://arxiv.org/abs/2406.16232)). This is useful when clauses pull the parameters in opposite directions. The training loop is unchanged; only the `aggregator` argument differs.

This feature is experimental and requires `pip install "sltn[multiobjective]"`.

In [ ]:
try:
    import torchjd  # noqa: F401
except ModuleNotFoundError:
    print('Skipping multi-objective training. To run this example: pip install "sltn[multiobjective]"')
else:
    torch.manual_seed(0)
    interp_jd = train(new_interp(), aggregator="upgrad")
    interp_jd["x_pos"], interp_jd["x_neg"] = pos_tr.to(device), neg_tr.to(device)
    print("clause satisfaction:", {k: round(v, 3) for k, v in kb_eval(interp_jd, kb_dict).items()})
    print(f"test accuracy (UPGrad): {accuracy(interp_jd):.2f}")

## C.2 — Training with PyTorch Lightning

[PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/) is a framework that separates the research code (model, loss, optimiser) from the engineering code. The user implements a `LightningModule` with `training_step` (one batch → loss) and `configure_optimizers`; a `Trainer` then handles the optimisation loop, device placement, mixed precision, multi-GPU and multi-node training, logging to TensorBoard, Weights & Biases and other backends, checkpointing, early stopping and reproducibility. Lightning is optional: everything in Tutorials 1–5 works with plain PyTorch.

Since an `Interpretation` is an `nn.Module`, it can be used directly as a submodule of a `LightningModule`: `training_step` grounds the batch and returns `1 − sat`, and `configure_optimizers` uses `interp.parameters()`. Lightning then switches the interpretation between training and evaluation modes, moves it to the selected device and includes its weights in its checkpoints. Install it with `pip install "sltn[lightning]"`.

The `Trainer` expects a single dataloader, so the two class loaders are wrapped in an iterable that yields `(x_pos, x_neg)` pairs.

> **Multi-objective optimisation with Lightning.** With a TorchJD aggregator, set `self.automatic_optimization = False` and, inside `training_step`, call `opt.zero_grad()`, `kb_backward(opt, self.interp, kb_dict, aggregator=...)` and `opt.step()`, with `opt = self.optimizers()`.

In [ ]:
try:
    import lightning as L
except ModuleNotFoundError:
    print('Skipping Lightning. To run this example: pip install "sltn[lightning]"')
else:

    class PairedLoader:
        """Yields (x_pos, x_neg) batches; Lightning needs one iterable."""
        def __iter__(self):
            return ((p, n) for (p,), (n,) in zip(loader_pos, loader_neg))
        def __len__(self):
            return min(len(loader_pos), len(loader_neg))

    class KBLearner(L.LightningModule):
        def __init__(self, interp, kb):
            super().__init__()
            self.interp, self.kb = interp, kb

        def training_step(self, batch, batch_idx):
            self.interp["x_pos"], self.interp["x_neg"] = batch
            loss = 1 - self.interp(self.kb).value
            self.log("loss", loss, prog_bar=True, batch_size=len(batch[0]))
            return loss

        def configure_optimizers(self):
            return torch.optim.Adam(self.interp.parameters(), lr=1e-2)

    interp_pl = Interpretation(sig)
    interp_pl["Point"] = sltn.Type("R2", shape=(2,))
    interp_pl["InDisc"] = InDiscNet()             # on the CPU: Lightning moves it

    trainer = L.Trainer(max_epochs=EPOCHS, accelerator="auto", logger=False,
                        enable_checkpointing=False, enable_progress_bar=False,
                        enable_model_summary=False)
    trainer.fit(KBLearner(interp_pl, kb), train_dataloaders=PairedLoader())

    interp_pl.to(device)
    print(f"test accuracy (Lightning): {accuracy(interp_pl):.2f}")

# Part D — Evaluation and debugging

This part reuses the animals of Tutorials 1–2: an animal is a vector in ℝ² (size, furriness), and `IsDog` / `IsCat` are true near a dog and a cat prototype.

In [ ]:
sig = Signature("animals")
sig.sort("Animal")
sig.variable("x", "Animal")
sig.predicate("IsDog", ["Animal"])
sig.predicate("IsCat", ["Animal"])

interp = Interpretation(sig).to(device)
interp["Animal"] = sltn.Type("R2", shape=(2,))   # (size, furriness)
dog = torch.tensor([0.0, 0.0], device=device)
cat = torch.tensor([1.0, 1.0], device=device)
interp["IsDog"] = lambda a: torch.exp(-torch.norm(a - dog, dim=-1, keepdim=True))
interp["IsCat"] = lambda a: torch.exp(-torch.norm(a - cat, dim=-1, keepdim=True))
interp["x"] = torch.randn(8, 2, device=device)

## D.1 — Classical Boolean evaluation (`Logic.classical()`)

Training requires smooth, differentiable operators so that gradients reach every clause. For evaluation and debugging, however, the crisp question is often more informative: *if every predicate is thresholded at 0.5, does the formula hold in classical two-valued first-order logic?*

`Logic.classical()` builds a Boolean operator bundle (`min`, `max`, `1 - x`, Gödel implication) and binarises every `Bool`-typed atom as `(p >= threshold).float()` (threshold 0.5 by default). Passed to `interp.with_updates(logic=...)`, it yields a crisp copy that shares all groundings and leaves `interp` unchanged:

- `Logic.classical()`: crisp Boolean evaluation, with atoms thresholded at 0.5;
- `Logic.classical(threshold=0.8)`: a custom threshold;
- `Logic.classical(threshold=None)`: Zadeh/Gödel logic without thresholding (`min` / `max` applied directly to the values in [0, 1]).

In [ ]:
crisp        = interp.with_updates(logic=Logic.classical())
strict_crisp = interp.with_updates(logic=Logic.classical(threshold=0.8))
fuzzy_godel  = interp.with_updates(logic=Logic.classical(threshold=None))

print("IsDog(x) soft         :", interp("IsDog(x)").value.squeeze(-1).round(decimals=3))
print("IsDog(x) classical    :", crisp("IsDog(x)").value.squeeze(-1))

for formula in ["exists x: IsDog(x)", "forall x: (IsDog(x) or IsCat(x))"]:
    f = sig.parse(formula)
    print(f"\n{formula}")
    print(f"  fuzzy default           : {interp(f).value.item():.4f}")
    print(f"  Zadeh/Gödel (no thresh) : {fuzzy_godel(f).value.item():.4f}")
    print(f"  classical (thr = 0.5)   : {crisp(f).value.item():.1f}")
    print(f"  classical (thr = 0.8)   : {strict_crisp(f).value.item():.1f}")

## D.2 — Checking truth values: `validate_type_constraint`

Every denotation has a `Type`, and a `Type` can carry a constraint. The denotation of a formula, of type `Bool`, must lie in [0, 1]. Tutorial 1 (section 2.6) shows the error raised when a predicate returns a distance instead of a truth degree. The `validate_type_constraint` argument of `Interpretation` determines which denotations are checked.

| Value | Checks | Use |
|---|---|---|
| `"groundings"` (default) | the denotations computed directly by a grounding: variables, constants, applications of functions and predicates, structural relations | always, since it detects a wrong grounding where it is applied, at little cost |
| `"all"` | also the denotation of every connective and quantifier | to debug a custom operator, or to find the first subformula whose denotation contains a NaN |
| `"off"` | nothing | for speed, once the groundings are trusted |

The mode is set in the constructor, `Interpretation(sig, validate_type_constraint="all")`, or later as an attribute, and is preserved by `copy()`, `deepcopy()` and `with_updates()`. On a GPU, `"all"` is slow, because each check synchronises with the device.

Below, `Close` returns a similarity that exceeds 1, and a custom conjunction `a + b` leaves [0, 1] although both of its inputs are valid. With `"off"`, neither error is detected; `"groundings"` detects the predicate; only `"all"` detects the connective. Otherwise, the out-of-range values would be silently aggregated by `∀`.

In [ ]:
sig.predicate("Close", ["Animal"])
interp["Close"] = lambda a: 2 * torch.exp(-torch.norm(a - dog, dim=-1, keepdim=True))   # up to 2: not a truth degree

def check(i, formula):
    for mode in ["off", "groundings", "all"]:
        i.validate_type_constraint = mode
        try:
            print(f"  {mode:<10s} {i(formula).value.item():.3f}")
        except ValueError as e:
            print(f"  {mode:<10s} ValueError: {e}")

print("forall x: Close(x)")
check(interp, "forall x: Close(x)")

custom = interp.copy()
custom.logic["&"] = lambda a, b: a + b                  # not a t-norm: can exceed 1
print("\nforall x: (IsDog(x) & IsCat(x)), with a + b as ∧")
check(custom, "forall x: (IsDog(x) & IsCat(x))")
interp.validate_type_constraint = "groundings"